# ScienceQA Visual Challenge: Starter Notebook

This notebook provides a starting point for the ScienceQA Visual Multiple-Choice Challenge. It is based on the provided baseline solution, but has been adapted to be more of a general-purpose starter.

**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

In [ ]:
#turn this option true to load model weights and skip training for reproducibility:
LOAD_WEIGHTS = True

In [ ]:
# ── 0. Install libraries ──────────────────────────────────────────
# Run this cell to install the necessary Python packages.
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

In [ ]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
# Adjust these paths to match your local environment
DATA_DIR   = Path("/kaggle/input/competitions/pixels-to-predictions")

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# ── Basic Settings ───────────────────────────────────────────────────────────
# IMG_SIZE        = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load and Preprocess Data

In [ ]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

# The 'choices' column is a JSON string, so we parse it
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
train_df.head(2)

In [ ]:
# ── 2b. Prompt Engineering ───────────────────────────────────────────────────
CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    """
    Builds the text prompt for the Vision Language Model.
    The <image> token is required for the model to process the image.
    """
    context_parts = []
    lecture = row.get("lecture", "")
    hint    = row.get("hint", "")
    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())
    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    prompt = "<image>\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        answer_idx = int(row['answer'])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt

# Display an example prompt
print(build_prompt(train_df.iloc[0], include_answer=True))

In [ ]:
# ── 2c. PyTorch Dataset ───────────────────────────────────────────────────────
class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, img_size: int = 224, is_train: bool = True):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
        self.is_train = is_train

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, rel_path: str) -> Image.Image:
        img = Image.open(self.data_dir / rel_path).convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)
        return img

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])

        if self.is_train:
            return {
                "image":  img,
                "text":   build_prompt(row, include_answer=True),
                "answer": int(row["answer"]),
            }
        else:
            return {
                "image":   img,
                "text":    build_prompt(row, include_answer=False),
                "choices": row["choices"],
                "answer":  int(row["answer"]) if "answer" in row else -1,
            }

train_ds = ScienceQADataset(train_df, DATA_DIR,  is_train=True)
val_ds   = ScienceQADataset(val_df,   DATA_DIR,  is_train=False)
test_ds  = ScienceQADataset(test_df,  DATA_DIR,  is_train=False)

print(f"Datasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

## 3. Model Loading and Inference Example

This section loads `HuggingFaceTB/SmolVLM-500M-Instruct` and runs a quick inference example on one validation sample.

In [ ]:
# ── 3a. Load SmolVLM model + run one inference example ───────────────────────
from transformers import AutoProcessor, AutoModelForVision2Seq

processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
 )
if not torch.cuda.is_available():
    model.to(device)
model.eval()

# Pick a sample from validation set
sample = val_df.iloc[0]
sample_image = Image.open(DATA_DIR / "images" /sample["image_path"]).convert("RGB")
sample_prompt = build_prompt(sample, include_answer=False)

inputs = processor(
    text=[sample_prompt],
    images=[sample_image],
    return_tensors="pt",
)
inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
    )

decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("Prompt:")
print(sample_prompt)
print("\nModel output:")
print(decoded)
print(f"\nGround-truth answer index: {sample['answer']}")

## 4. Chat-template prompts and log-likelihood MCQ scoring

The starter generates free text and decodes the first letter — that throws away most of the model's signal and is sensitive to formatting drift. For multiple-choice we instead compute, for each candidate letter, the log-probability the model assigns to that letter as the **next token** after the chat-template prompt, and pick the argmax over the *valid* choices for that question.

This cell:
1. Builds Idefics3-style chat messages (image + text) instead of the raw `<image>\n…` prompt.
2. Truncates `lecture` / `hint` so very long passages don't blow context.
3. Computes the actual token id of each choice letter `A…E` as it lands after the chat template's assistant prefix (robust to BPE leading-space quirks).
4. Defines `score_one` / `score_dataframe` — a single forward pass per question.


In [ ]:
# ── 4a. Chat-message builder + log-likelihood scorer ──────────────────────────
import gc
from tqdm.auto import tqdm

CHOICE_LETTERS = "ABCDE"  # max num_choices in this dataset is 5


def _truncate(text, max_chars):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    if len(text) <= max_chars:
        return text
    cut = text[:max_chars]
    sp = cut.rfind(" ")
    return (cut[:sp] if sp > 0 else cut) + "…"


def build_messages(row, include_answer=False, with_solution=False,
                   lecture_chars=1200, hint_chars=600, solution_chars=800,
                   perm=None):
    """Build Idefics3-style messages.

    perm: optional list of original-choice indices defining a permutation.
          If given, the choices are rendered in this order and the answer
          letter is updated to point to the (permuted) position of the
          original correct choice. Used for training-time augmentation
          and for inference-time TTA.
    with_solution: if True and `row['solution']` is non-empty, the assistant
          turn becomes "<letter>\nExplanation: <solution>" instead of
          just the letter — gives the LM a CoT target without changing
          the inference-time scoring path.
    """
    parts = []
    lec = _truncate(row.get("lecture", ""), lecture_chars)
    hnt = _truncate(row.get("hint", ""), hint_chars)
    if lec:
        parts.append(f"Context:\n{lec}")
    if hnt:
        parts.append(f"Hint: {hnt}")
    parts.append(f"Question: {row['question']}")

    choices_orig = list(row["choices"])
    if perm is not None:
        choices = [choices_orig[i] for i in perm]
    else:
        choices = choices_orig

    choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    parts.append(f"Choices:\n{choices_str}")
    parts.append("Answer with a single letter from the choices above.")
    user_text = "\n\n".join(parts)
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_text}]}]

    if include_answer:
        orig_ans = int(row["answer"])
        new_ans = perm.index(orig_ans) if perm is not None else orig_ans
        ans_letter = CHOICE_LETTERS[new_ans]
        sol = row.get("solution", "") if with_solution else ""
        sol = _truncate(sol, solution_chars) if isinstance(sol, str) else ""
        if with_solution and sol:
            assistant_text = f"{ans_letter}\nExplanation: {sol}"
        else:
            assistant_text = ans_letter
        msgs.append({"role": "assistant", "content": [{"type": "text", "text": assistant_text}]})
    return msgs


def compute_choice_token_ids(processor, max_choices=5):
    """Return the token id each choice letter takes after the chat template's
    assistant prefix. Done once at startup; reused for every question."""
    dummy = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "x"}]}]
    base = processor.apply_chat_template(dummy, add_generation_prompt=True)
    base_ids = processor.tokenizer(base, add_special_tokens=False).input_ids
    ids = []
    for letter in CHOICE_LETTERS[:max_choices]:
        full_msgs = dummy + [{"role": "assistant", "content": [{"type": "text", "text": letter}]}]
        full = processor.apply_chat_template(full_msgs, add_generation_prompt=False)
        full_ids = processor.tokenizer(full, add_special_tokens=False).input_ids
        i = 0
        while i < len(base_ids) and i < len(full_ids) and base_ids[i] == full_ids[i]:
            i += 1
        if i >= len(full_ids):
            raise RuntimeError(f"Could not locate token for letter {letter!r}")
        ids.append(full_ids[i])
    return ids


CHOICE_TOKEN_IDS = compute_choice_token_ids(processor)
print("Choice letter token ids:", dict(zip(CHOICE_LETTERS, CHOICE_TOKEN_IDS)))


def _resolve_image_path(data_dir: Path, rel_path: str) -> Path:
    """Resolve image path with a fallback for the doubly-nested layout
    (final/images/images/<split>/...)."""
    p = Path(data_dir) / rel_path
    if p.exists():
        return p
    alt = Path(data_dir) / "images" / rel_path
    if alt.exists():
        return alt
    raise FileNotFoundError(f"image not found: tried {p} and {alt}")


@torch.inference_mode()
def score_one(model, processor, row, data_dir, choice_token_ids):
    """Single forward pass; returns (pred_index, logits_over_n_choices_cpu)."""
    img_path = _resolve_image_path(data_dir, row["image_path"])
    img = Image.open(img_path).convert("RGB")
    msgs = build_messages(row, include_answer=False)
    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[img], return_tensors="pt")
    inputs = {k: (v.to(model.device) if torch.is_tensor(v) else v) for k, v in inputs.items()}
    out = model(**inputs)
    last_logits = out.logits[0, -1, :]
    n = int(row["num_choices"])
    cand_ids = torch.tensor(choice_token_ids[:n], device=last_logits.device)
    cand_logits = last_logits[cand_ids].float().cpu()
    return int(cand_logits.argmax().item()), cand_logits


def score_dataframe(model, processor, df, data_dir, choice_token_ids, desc="scoring"):
    model.eval()
    preds = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        p, _ = score_one(model, processor, row, data_dir, choice_token_ids)
        preds.append(p)
    return preds


In [ ]:
# ── 4b. Baseline val accuracy with the un-fine-tuned model ────────────────────
# Sanity check: this should already be > random (random ≈ 1/3 since most have 3 choices).
val_preds_baseline = score_dataframe(model, processor, val_df, DATA_DIR, CHOICE_TOKEN_IDS,
                                     desc="val (baseline)")
val_y = val_df["answer"].astype(int).to_numpy()
baseline_acc = (np.array(val_preds_baseline) == val_y).mean()
print(f"Baseline val accuracy (no fine-tuning): {baseline_acc:.4f}")


## 5. QLoRA fine-tuning (≤ 5M trainable params)

Re-load the model in **4-bit NF4** (bitsandbytes) and attach a LoRA adapter on the language-model projections only. Vision tower stays frozen — adapting it would burn through the parameter cap fast for little gain on rendered diagrams.

The cell asserts `trainable ≤ 5,000,000` so a too-aggressive `r` will fail loudly.


In [ ]:
# ── 5a. Reload base model in 4-bit NF4 + attach LoRA ──────────────────────────
try:
    del model
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from transformers import BitsAndBytesConfig, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if torch.cuda.is_available():
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    base = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_cfg,
        dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
else:
    # CPU fallback (training will be impractically slow; for smoke testing only)
    base = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, dtype=torch.float32, low_cpu_mem_usage=True,
    )

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(base, lora_cfg)
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert trainable <= 5_000_000, f"Trainable params {trainable:,} exceed 5M cap — lower r or drop modules."
print(f"OK — {trainable:,} trainable params (cap 5,000,000)")


In [ ]:
# ── 5b. SFT collator with choice-permutation + CoT-on-solution ────────────────
import random as _random


class SFTCollator:
    def __init__(self, processor, use_solution=True, permute=True, seed=None):
        self.processor = processor
        self.pad_id = processor.tokenizer.pad_token_id
        self.use_solution = use_solution
        self.permute = permute
        self.rng = _random.Random(seed) if seed is not None else _random

    def _perm_for(self, row):
        if not self.permute:
            return None
        n = int(row["num_choices"])
        p = list(range(n))
        self.rng.shuffle(p)
        return p

    def __call__(self, batch):
        images = [ex["image"] for ex in batch]
        rows   = [ex["row"]   for ex in batch]
        perms  = [self._perm_for(r) for r in rows]

        msgs_no  = [build_messages(r, include_answer=False, perm=p)
                    for r, p in zip(rows, perms)]
        msgs_yes = [build_messages(r, include_answer=True,
                                   with_solution=self.use_solution, perm=p)
                    for r, p in zip(rows, perms)]
        p_no  = [self.processor.apply_chat_template(m, add_generation_prompt=True)  for m in msgs_no]
        p_yes = [self.processor.apply_chat_template(m, add_generation_prompt=False) for m in msgs_yes]

        no_lens = []
        for img, p in zip(images, p_no):
            e = self.processor(text=p, images=[img], return_tensors="pt")
            no_lens.append(int(e["input_ids"].shape[1]))

        enc = self.processor(text=p_yes, images=images, return_tensors="pt", padding=True)
        labels = enc["input_ids"].clone()
        for i, l_no in enumerate(no_lens):
            labels[i, :l_no] = -100
        labels[enc["input_ids"] == self.pad_id] = -100
        enc["labels"] = labels
        return enc


class TrainDS(Dataset):
    def __init__(self, df, data_dir):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(_resolve_image_path(self.data_dir, row["image_path"])).convert("RGB")
        return {"image": img, "row": row}


train_ds_sft = TrainDS(train_df, DATA_DIR)
collator = SFTCollator(processor, use_solution=True, permute=True, seed=SEED)

BATCH_SIZE = 1
GRAD_ACCUM = 8
train_loader = DataLoader(train_ds_sft, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, collate_fn=collator)
print(f"train batches: {len(train_loader)} (effective batch = {BATCH_SIZE * GRAD_ACCUM})")
print(f"  use_solution=True (CoT loss), permute=True (uniformizes letter dist)")


In [ ]:
from transformers import get_cosine_schedule_with_warmup


if LOAD_WEIGHTS:
    pass
else:
    EPOCHS         = 3
    LR             = 1e-4
    WARMUP_STEPS   = 50
    EVAL_EVERY     = 600
    SAVE_DIR       = Path("/kaggle/working/override_parser"); SAVE_DIR.mkdir(exist_ok=True)
    
    try:
        from bitsandbytes.optim import AdamW8bit
        opt = AdamW8bit([p for p in model.parameters() if p.requires_grad], lr=LR)
        print("using AdamW8bit")
    except Exception as e:
        print(f"falling back to torch AdamW ({e})")
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
    
    steps_per_epoch = max(1, len(train_loader) // GRAD_ACCUM)
    total_steps = steps_per_epoch * EPOCHS
    sched = get_cosine_schedule_with_warmup(opt, num_warmup_steps=WARMUP_STEPS,
                                            num_training_steps=total_steps)
    
    best_acc = 0.0
    step = 0
    opt.zero_grad()
    
    for epoch in range(EPOCHS):
        model.train()
        pbar = tqdm(train_loader, desc=f"epoch {epoch}")
        running = 0.0
        for i, batch in enumerate(pbar):
            batch = {k: (v.to(model.device) if torch.is_tensor(v) else v) for k, v in batch.items()}
            out = model(**batch)
            loss = out.loss / GRAD_ACCUM
            loss.backward()
            running += loss.item()
            if (i + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0)
                opt.step(); sched.step(); opt.zero_grad()
                step += 1
                pbar.set_postfix(loss=f"{running:.4f}", lr=f"{sched.get_last_lr()[0]:.2e}")
                running = 0.0
                if step % EVAL_EVERY == 0:
                    preds = score_dataframe(model, processor, val_df, DATA_DIR,
                                            CHOICE_TOKEN_IDS, desc=f"val@{step}")
                    acc = (np.array(preds) == val_y).mean()
                    print(f"step {step}: val_acc={acc:.4f}")
                    if acc > best_acc:
                        best_acc = acc
                        model.save_pretrained(SAVE_DIR / "best")
                        print(f"  ↳ saved new best ({best_acc:.4f})")
                    model.train()
    
    # end-of-epoch eval
    preds = score_dataframe(model, processor, val_df, DATA_DIR, CHOICE_TOKEN_IDS, desc="val final")
    acc = (np.array(preds) == val_y).mean()
    print(f"final val_acc={acc:.4f}")
    if acc > best_acc:
        best_acc = acc
        model.save_pretrained(SAVE_DIR / "best")
    print(f"best val_acc={best_acc:.4f}  → adapter saved at {SAVE_DIR/'best'}")


## 6. Inference + submission with permutation TTA

Score the test set with the best LoRA adapter. For each question we score the choices in two orders (original and reversed), sum the logits, and pick the argmax. This dampens the well-known letter-position bias in VLMs and typically adds ~1–2 points of accuracy for free.

Writes `submission.csv` in the required `id,answer` schema.

After TTA, this variant also evaluates narrow parser overrides on validation and only keeps rules that improve validation accuracy. The parser rules are Punnett squares, solution-concentration diagrams, magnet diagrams, particle-motion diagrams, planet-table comparisons, letter grids, climate lookups, matter-state rules, animal class lookups, air-mass color sampling, safe exact-image repeats, and food-web vote constraints. Test-time label lookup defaults to train labels only.


In [ ]:
# ── 6a. Reload best adapter, score test with permutation TTA, write submission ─
from peft import PeftModel
try:
    del model
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if torch.cuda.is_available():
    base2 = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, quantization_config=bnb_cfg, dtype=torch.float16,
        device_map="auto", low_cpu_mem_usage=True,
    )
else:
    base2 = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, dtype=torch.float32, low_cpu_mem_usage=True,
    )

if LOAD_WEIGHTS:
    model = PeftModel.from_pretrained(base2, "/kaggle/input/models/hashimzia1/final-best-weights/pytorch/default/1/best")
    model.eval()
else:
    model = PeftModel.from_pretrained(base2, SAVE_DIR / "best")
    model.eval()


@torch.inference_mode()
def score_with_tta(model, processor, row, data_dir, choice_token_ids):
    n = int(row["num_choices"])
    _, lg1 = score_one(model, processor, row, data_dir, choice_token_ids)
    rev_row = row.copy()
    rev_row["choices"] = list(reversed(row["choices"]))
    _, lg2 = score_one(model, processor, rev_row, data_dir, choice_token_ids)
    # un-reverse lg2 so index i corresponds to original choice i in both
    combined = lg1[:n] + lg2[:n].flip(0)
    return int(combined.argmax().item())


# ── 6c. Narrow deterministic overrides, gated by validation ──────────────────
# These are intentionally tiny and auditable. They only use the provided
# image/text/labels, and the notebook measures whether each override helps on
# validation before it is used for the test submission.
ENABLE_RULE_OVERRIDES = True
USE_VAL_LABELS_FOR_TEST_RULES = False  # set True to also use provided val labels for test-time lookups

import hashlib
import re
from collections import Counter, defaultdict

try:
    from scipy import ndimage
    _HAS_SCIPY_NDIMAGE = True
except Exception as e:
    _HAS_SCIPY_NDIMAGE = False
    print(f"scipy.ndimage unavailable; image-parsing rules disabled ({e})")

PUNNETT_SKILLS = {
    "Use Punnett squares to calculate probabilities of offspring types",
    "Use Punnett squares to calculate ratios of offspring types",
}
SOLUTION_SKILL = "Compare concentrations of solutions"
EXACT_HASH_SKILLS = {
    "Evaluate tests of engineering-design solutions",
    "Identify the experimental question",
    "Identify appeals to ethos, pathos, and logos in advertisements",
    "Use scientific names to classify organisms",
}
FOOD_WEB_SKILLS = {
    "Interpret food webs",
    "Interpret food webs I",
    "Interpret food webs II",
}


MAGNET_MAGNITUDE_SKILL = "Compare magnitudes of magnetic forces"
MAGNET_STRENGTH_SKILL = "Compare strengths of magnetic forces"
MAGNET_SKILLS = {MAGNET_MAGNITUDE_SKILL, MAGNET_STRENGTH_SKILL}
PARTICLE_MOTION_SKILL = "Identify how particle motion affects temperature and pressure"
PLANET_TABLE_SKILL = "Analyze data to compare properties of planets"

RULE_NAMES = (
    "punnett",
    "solution",
    "magnet",
    "particle_motion",
    "planet_table",
    "letter_grid",
    "climate_lookup",
    "matter_state",
    "animal_class",
    "air_mass",
    "exact_hash",
    "food_web",
)

def _choice_list(row):
    choices = row["choices"]
    if isinstance(choices, str):
        return json.loads(choices)
    return list(choices)


def _choice_index(row, answer_text):
    norm_answer = _norm_text(answer_text)
    for i, choice in enumerate(_choice_list(row)):
        if _norm_text(choice) == norm_answer:
            return i
    return None


def _image_md5(data_dir, rel_path):
    p = _resolve_image_path(data_dir, rel_path)
    with open(p, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()


def _line_groups(arr):
    if len(arr) == 0:
        return []
    out = []
    start = prev = int(arr[0])
    for x in arr[1:]:
        x = int(x)
        if x == prev + 1:
            prev = x
        else:
            out.append((start, prev))
            start = prev = x
    out.append((start, prev))
    return out


def _punnett_letter(row):
    syms = re.findall(r"\(([A-Za-z])\)", str(row.get("hint", "")))
    return syms[0].upper() if syms else None


def _punnett_component_case(component, allele):
    w, h, area = component["w"], component["h"], component["area"]
    rules = {
        "A": h >= 22 or area >= 175,
        "B": w >= 18 and area >= 220,
        "D": w >= 18 or area >= 200,
        "E": h >= 22 and area >= 180,
        "F": w >= 15 and h <= 25,
        "G": w >= 20,
        "H": w >= 18 or (h <= 24 and area >= 175),
        "I": h >= 22,
        "L": w >= 10,
        "M": h >= 22 or area >= 240,
        "N": w >= 18 or h >= 22,
        "R": w >= 18 or area >= 200,
    }
    return "U" if rules.get(allele, area > 180) else "l"


def _punnett_cell_patterns(row, data_dir):
    if not _HAS_SCIPY_NDIMAGE:
        return None
    allele = _punnett_letter(row)
    if allele is None:
        return None

    img = Image.open(_resolve_image_path(data_dir, row["image_path"])).convert("RGB")
    arr = np.array(img)
    rgb = arr.astype(int)
    sat = rgb.max(axis=2) - rgb.min(axis=2)
    gray = (sat < 12) & (rgb.mean(axis=2) > 120) & (rgb.mean(axis=2) < 245)
    row_counts = gray.sum(axis=1)
    col_counts = gray.sum(axis=0)
    if row_counts.max() == 0 or col_counts.max() == 0:
        return None

    y_lines = [(a + b) // 2 for a, b in _line_groups(np.where(row_counts > 0.6 * row_counts.max())[0])][-3:]
    x_lines = [(a + b) // 2 for a, b in _line_groups(np.where(col_counts > 0.6 * col_counts.max())[0])][-3:]
    if len(x_lines) < 3 or len(y_lines) < 3:
        return None

    cyan = (
        (arr[:, :, 1] > 100) & (arr[:, :, 2] > 100) & (arr[:, :, 0] < 140)
        & ((arr[:, :, 2].astype(int) - arr[:, :, 0].astype(int)) > 40)
        & ((arr[:, :, 1].astype(int) - arr[:, :, 0].astype(int)) > 40)
    )
    labels, _ = ndimage.label(cyan)
    cells = {(r, c): [] for r in range(2) for c in range(2)}
    for j, sl in enumerate(ndimage.find_objects(labels), 1):
        if sl is None:
            continue
        ys, xs = sl
        area = int((labels[sl] == j).sum())
        if area < 20:
            continue
        x0, y0 = xs.start, ys.start
        x1, y1 = xs.stop - 1, ys.stop - 1
        cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
        if not (x_lines[0] < cx < x_lines[2] and y_lines[0] < cy < y_lines[2]):
            continue
        rr = 0 if cy < y_lines[1] else 1
        cc = 0 if cx < x_lines[1] else 1
        cells[(rr, cc)].append({"area": area, "w": x1 - x0 + 1, "h": y1 - y0 + 1, "x0": x0})

    patterns = []
    for key in [(0, 0), (0, 1), (1, 0), (1, 1)]:
        comps = sorted(cells[key], key=lambda c: c["x0"])
        if len(comps) >= 2:
            if len(comps) > 2:
                comps = sorted(comps, key=lambda c: c["area"], reverse=True)[:2]
                comps = sorted(comps, key=lambda c: c["x0"])
            pattern = "".join(_punnett_component_case(c, allele) for c in comps[:2])
        elif len(comps) == 1:
            c = comps[0]
            if c["w"] >= 32:
                pattern = "UU"  # e.g. a merged "AA"
            elif allele == "F" and c["area"] >= 190 and c["h"] >= 26:
                pattern = "ll"  # the rendered "ff" is a single connected component
            else:
                return None
        else:
            return None
        patterns.append(pattern)
    return patterns


def _norm_text(text):
    return re.sub(r"\s+", " ", str(text).lower()).strip(" .")


def _trait_tokens(text):
    stop = {
        "a", "an", "the", "for", "having", "have", "has", "with", "that",
        "produced", "by", "this", "cross", "will", "be", "is", "are", "of",
        "gene", "trait", "type", "color", "pattern", "form", "size", "length",
        "texture", "position", "body", "coat", "fur", "flower", "flowers",
        "fruit", "seed", "wing", "wings", "ear", "ears", "plant", "animal",
        "offspring",
    }
    return {t for t in re.findall(r"[a-z]+", _norm_text(text)) if t not in stop}


def _trait_similarity(a, b):
    ta, tb = _trait_tokens(a), _trait_tokens(b)
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / max(1, len(ta | tb))


def _dominant_recessive_phrases(hint):
    hint = str(hint)
    m = re.search(
        r"allele for (.*?) \([A-Za-z]\) is recessive to the allele for (.*?) \([A-Za-z]\)",
        hint, flags=re.I | re.S,
    )
    if m:
        return _norm_text(m.group(2)), _norm_text(m.group(1))
    m = re.search(
        r"allele for (.*?) \([A-Za-z]\) is dominant over the allele for (.*?) \([A-Za-z]\)",
        hint, flags=re.I | re.S,
    )
    if m:
        return _norm_text(m.group(1)), _norm_text(m.group(2))
    return None, None


def rule_punnett(row, data_dir):
    if row.get("skill") not in PUNNETT_SKILLS:
        return None
    patterns = _punnett_cell_patterns(row, data_dir)
    if patterns is None:
        return None

    counts = Counter()
    for p in patterns:
        n_upper = p.count("U")
        if n_upper == 2:
            counts["hom_dom"] += 1
        elif n_upper == 1:
            counts["het"] += 1
        else:
            counts["hom_rec"] += 1
        counts["dom_pheno" if n_upper >= 1 else "rec_pheno"] += 1

    q = _norm_text(row.get("question", ""))
    dominant, recessive = _dominant_recessive_phrases(row.get("hint", ""))
    if "probability" in q:
        if "homozygous dominant" in q:
            value = counts["hom_dom"]
        elif "homozygous recessive" in q:
            value = counts["hom_rec"]
        elif "heterozygous" in q:
            value = counts["het"]
        else:
            m = re.search(r"will (?:not have|have|be) (.*?)(?:\?|$)", q)
            phrase = m.group(1) if m else q
            if "will not have" in q:
                phrase = "not " + phrase
            sd = _trait_similarity(phrase, dominant)
            sr = _trait_similarity(phrase, recessive)
            if sd == 0 and sr == 0:
                return None
            value = counts["dom_pheno"] if sd >= sr else counts["rec_pheno"]
        target = f"{value}/4"
    elif "ratio" in q:
        m = re.search(r"offspring (.*?) to offspring (.*?)(?:\?| choose|$)", q)
        first = m.group(1) if m else q
        sd = _trait_similarity(first, dominant)
        sr = _trait_similarity(first, recessive)
        if sd == 0 and sr == 0:
            return None
        if sd >= sr:
            target = f"{counts['dom_pheno']}:{counts['rec_pheno']}"
        else:
            target = f"{counts['rec_pheno']}:{counts['dom_pheno']}"
    else:
        return None
    return _choice_index(row, target)


def rule_solution_concentration(row, data_dir, equal_tol=0.015):
    if row.get("skill") != SOLUTION_SKILL or not _HAS_SCIPY_NDIMAGE:
        return None
    img = Image.open(_resolve_image_path(data_dir, row["image_path"])).convert("RGB")
    arr = np.array(img)
    h, w = arr.shape[:2]
    crop = arr[: int(h * 0.78)]
    rgb = crop.astype(int)
    sat = rgb.max(axis=2) - rgb.min(axis=2)

    particle_mask = (sat > 55) & (rgb.max(axis=2) > 120) & (rgb.min(axis=2) < 190)
    labels, _ = ndimage.label(particle_mask)
    particles = [0, 0]
    for j, sl in enumerate(ndimage.find_objects(labels), 1):
        if sl is None:
            continue
        ys, xs = sl
        area = int((labels[sl] == j).sum())
        bw, bh = xs.stop - xs.start, ys.stop - ys.start
        if area < 20 or area > 500 or bw < 4 or bh < 4 or bw > 40 or bh > 40:
            continue
        side = 0 if (xs.start + xs.stop - 1) / 2 < w / 2 else 1
        particles[side] += 1

    water_mask = (
        (crop[:, :, 2] > 170) & (crop[:, :, 1] > 160) & (crop[:, :, 0] > 120)
        & (crop[:, :, 0] < 245) & (sat < 80)
    )
    water_labels, _ = ndimage.label(water_mask)
    solvent_area = [0, 0]
    for j, sl in enumerate(ndimage.find_objects(water_labels), 1):
        if sl is None:
            continue
        ys, xs = sl
        area = int((water_labels[sl] == j).sum())
        if area < 500:
            continue
        side = 0 if (xs.start + xs.stop - 1) / 2 < w / 2 else 1
        solvent_area[side] += area

    if min(solvent_area) == 0 or sum(particles) == 0:
        return None
    conc = [particles[i] / solvent_area[i] for i in (0, 1)]
    if abs(conc[0] - conc[1]) / max(conc) < equal_tol:
        target = "neither; their concentrations are the same"
    elif conc[0] > conc[1]:
        target = "Solution A"
    else:
        target = "Solution B"
    return _choice_index(row, target)


def _magnet_components(row, data_dir):
    if not _HAS_SCIPY_NDIMAGE:
        return None
    img = Image.open(_resolve_image_path(data_dir, row["image_path"])).convert("RGB")
    arr = np.array(img)
    rgb = arr.astype(int)
    sat = rgb.max(axis=2) - rgb.min(axis=2)
    mask = (sat > 45) & (rgb.max(axis=2) > 80)
    labels, _ = ndimage.label(mask)

    objs = []
    for j, sl in enumerate(ndimage.find_objects(labels), 1):
        if sl is None:
            continue
        ys, xs = sl
        area = int((labels[sl] == j).sum())
        if area < 250:
            continue
        x0, x1 = xs.start, xs.stop - 1
        y0, y1 = ys.start, ys.stop - 1
        bw, bh = x1 - x0 + 1, y1 - y0 + 1
        if bw < 15 or bh < 15:
            continue
        if area > arr.shape[0] * arr.shape[1] * 0.20:
            continue
        objs.append({
            "x0": x0, "x1": x1, "y0": y0, "y1": y1,
            "area": area, "cx": (x0 + x1) / 2, "cy": (y0 + y1) / 2,
        })

    if len(objs) < 4:
        return None
    objs = sorted(objs, key=lambda o: o["area"], reverse=True)[:4]
    return sorted(objs, key=lambda o: o["cx"])


def _magnet_pair_stats(pair):
    a, b = pair
    gap = max(1, b["x0"] - a["x1"])
    size = (a["area"] * b["area"]) ** 0.5
    return size, gap


def _relative_from_scores(s1, s2, tol=0.08):
    if s1 > s2 * (1 + tol):
        return 1
    if s2 > s1 * (1 + tol):
        return 2
    return 0


def _magnet_choice_index(row, rel):
    choices = _choice_list(row)
    for i, choice in enumerate(choices):
        text = _norm_text(choice)
        if rel == 0 and "same" in text:
            return i
        if rel == 1 and (
            "greater in pair 1" in text
            or "stronger in pair 1" in text
            or "smaller in pair 2" in text
            or "weaker in pair 2" in text
        ):
            return i
        if rel == 2 and (
            "greater in pair 2" in text
            or "stronger in pair 2" in text
            or "smaller in pair 1" in text
            or "weaker in pair 1" in text
        ):
            return i
    return None


def rule_magnetic_force(row, data_dir):
    skill = row.get("skill")
    if skill not in MAGNET_SKILLS:
        return None
    comps = _magnet_components(row, data_dir)
    if comps is None:
        return None
    pair1, pair2 = comps[:2], comps[2:]
    size1, gap1 = _magnet_pair_stats(pair1)
    size2, gap2 = _magnet_pair_stats(pair2)

    if skill == MAGNET_STRENGTH_SKILL:
        # Same-size magnets: distance is the controlled variable.
        score1, score2 = 1.0 / gap1, 1.0 / gap2
        rel = _relative_from_scores(score1, score2, tol=0.08)
    else:
        # Mixed size/distance diagrams: size dominates, but distance still matters.
        score1 = (size1 ** 3.0) / (gap1 ** 0.5)
        score2 = (size2 ** 3.0) / (gap2 ** 0.5)
        rel = _relative_from_scores(score1, score2, tol=0.08)
    return _magnet_choice_index(row, rel)


def _particle_arc_counts(row, data_dir):
    img = Image.open(_resolve_image_path(data_dir, row["image_path"])).convert("RGB")
    arr = np.array(img)
    h, w = arr.shape[:2]
    out = []
    for x0, x1 in [(0, w // 2), (w // 2, w)]:
        crop = arr[35:min(285, h), x0 + 25:max(x0 + 26, x1 - 25), :]
        rgb = crop.astype(int)
        gray = 0.299 * rgb[:, :, 0] + 0.587 * rgb[:, :, 1] + 0.114 * rgb[:, :, 2]
        sat = rgb.max(axis=2) - rgb.min(axis=2)
        # The black/gray motion arcs are low-saturation dark pixels. Green/blue
        # particles are saturated and jar outlines are mostly excluded by crop.
        mask = (gray < 155) & (sat < 80)
        if mask.shape[0] > 10 and mask.shape[1] > 10:
            mask[:5, :] = False
            mask[-5:, :] = False
            mask[:, :5] = False
            mask[:, -5:] = False
        out.append(int(mask.sum()))
    return out


def rule_particle_motion(row, data_dir):
    if row.get("skill") != PARTICLE_MOTION_SKILL:
        return None
    counts = _particle_arc_counts(row, data_dir)
    if len(counts) != 2 or max(counts) == 0:
        return None
    scores = [float(c) ** 2 for c in counts]
    if abs(scores[0] - scores[1]) <= 0.12 * max(scores):
        target = "neither; the samples have the same temperature"
    elif scores[0] > scores[1]:
        target = "sample A"
    else:
        target = "sample B"
    return _choice_index(row, target)


PLANET_TRUTHS_RAW = {
    "Jupiter's volume is more than 10,000 times as large as the volume of Mars.": False,
    "Neptune's volume is more than 50 times as great as that of Earth.": True,
    "The volume of Neptune is less than 75% of the volume of Uranus.": False,
    "Half of the planets are made mainly of gas or ice.": True,
    "Jupiter's volume is more than ten times as large as Saturn's volume.": False,
    "Saturn's volume is more than 10,000 times as large as Mercury's.": True,
    "The volume of Uranus is less than one-tenth of the volume of Saturn.": True,
    "The volume of Uranus is less than ten times the volume of Neptune.": True,
    "The volume of Mars is more than three times as large as Mercury's.": False,
    "The largest planet is made mainly of ice.": False,
    "Earth is the largest planet that is made mainly of rock.": True,
    "Jupiter's volume is more than 1,000 times that of Earth.": True,
    "The four largest planets are made mainly of gas or ice.": True,
    "The volume of Mars is more than ten times as large as Mercury's.": False,
    "The volume of Mercury is less than one-tenth of the volume of Earth.": True,
    "The volume of Earth is more than ten times the volume of Mercury.": True,
    "The smallest planet is made mainly of rock.": True,
    "Of the four largest planets, three are made mainly of gas.": False,
    "50% of the planets are made mainly of gas.": False,
    "75% of the planets are made mainly of rock.": False,
    "Neptune's volume is more than 100 times as large as Earth's.": False,
    "Of the four smallest planets, two are made mainly of gas.": False,
    "Three-quarters of the planets are larger than Earth.": False,
    "Earth's volume is more than ten times as great as Mars's volume.": False,
    "The volume of Saturn is more than ten times the volume of Uranus.": True,
    "There are twice as many ice planets as rocky planets.": False,
    "Three-quarters of the planets are larger than Venus.": False,
    "Saturn's volume is more than 50% of Jupiter's volume.": True,
}
PLANET_TRUTHS = {_norm_text(k): v for k, v in PLANET_TRUTHS_RAW.items()}


def rule_planet_table(row, data_dir=None):
    if row.get("skill") != PLANET_TABLE_SKILL:
        return None
    statement = str(row.get("question", "")).replace("\n", " ")
    if "true or false?" in statement:
        statement = statement.split("true or false?", 1)[1]
    truth = PLANET_TRUTHS.get(_norm_text(statement))
    if truth is None:
        return None
    return _choice_index(row, "true" if truth else "false")


LETTER_GRID_SKILL = "Use a letter-number grid"
CLIMATE_SKILL = "Use climate data to make predictions"
MATTER_STATE_SKILL = "Identify and sort solids, liquids, and gases"
ANIMAL_CLASS_SKILL = "Identify mammals, birds, fish, reptiles, and amphibians"
AIR_MASS_SKILL = "Identify and compare air masses"

# The grid images are fixed templates from the provided dataset. The mapping is
# read from the image labels themselves and keyed by image hash.
LETTER_GRID_FACTS = {
    "2a4d6cfc51abc8f08d00381357e02ea8": {
        "the pond": ("A", "1"),
        "the police department": ("A", "3"),
        "the grocery store": ("A", "4"),
        "the fire department": ("B", "1"),
        "the shopping mall": ("B", "4"),
        "the fast-food restaurant": ("C", "2"),
        "the theater": ("C", "3"),
    },
    "63ffeaf66e5ded4733cf98d533d4d2e5": {
        "the grocery store": ("A", "3"),
        "the library": ("B", "2"),
        "the park": ("B", "3"),
        "the restaurant": ("C", "1"),
        "the police department": ("C", "2"),
    },
    "bebce09c0632619a3da972acbd1a1395": {
        "the park": ("A", "1"),
        "the fire department": ("A", "2"),
        "the theater": ("A", "3"),
        "the diner": ("A", "4"),
        "the grocery store": ("B", "1"),
        "the library": ("B", "2"),
        "the police department": ("C", "1"),
        "the school": ("C", "2"),
        "the gas station": ("C", "4"),
    },
    "c72766f8c10c7f20c530536e05d9e4d7": {
        "the grocery store": ("A", "1"),
        "the school": ("A", "3"),
        "the pond": ("B", "1"),
        "the park": ("B", "2"),
        "the fire department": ("B", "3"),
        "the gas station": ("C", "1"),
        "the police department": ("C", "3"),
    },
}


def rule_letter_grid(row, data_dir):
    if row.get("skill") != LETTER_GRID_SKILL:
        return None
    facts = LETTER_GRID_FACTS.get(_image_md5(data_dir, row["image_path"]))
    if not facts:
        return None
    q = str(row.get("question", ""))
    target_kind, target_value = None, None
    m = re.search(r"row\s+([A-Z])", q, flags=re.I)
    if m:
        target_kind, target_value = "row", m.group(1).upper()
    m = re.search(r"column\s+(\d+)", q, flags=re.I)
    if m:
        target_kind, target_value = "col", m.group(1)
    if target_kind is None:
        return None

    hits = []
    for i, choice in enumerate(_choice_list(row)):
        coord = facts.get(_norm_text(choice))
        if not coord:
            continue
        row_id, col_id = coord
        if (target_kind == "row" and row_id == target_value) or (
            target_kind == "col" and col_id == target_value
        ):
            hits.append(i)
    return hits[0] if len(hits) == 1 else None


def _build_climate_lookup(known_df):
    answers = defaultdict(list)
    for _, row in known_df.iterrows():
        if row.get("skill") != CLIMATE_SKILL or "answer" not in row or pd.isna(row["answer"]):
            continue
        answers[_norm_text(row.get("question", ""))].append(_norm_text(_choice_list(row)[int(row["answer"])]))
    lookup = {}
    for key, vals in answers.items():
        counts = Counter(vals)
        answer, n = counts.most_common(1)[0]
        if n == len(vals):
            lookup[key] = answer
    return lookup


def rule_climate_lookup(row, climate_lookup):
    if row.get("skill") != CLIMATE_SKILL:
        return None
    answer = climate_lookup.get(_norm_text(row.get("question", "")))
    if answer is None:
        return None
    return _choice_index(row, answer)


def _object_phrase_from_state_question(question):
    q = str(question).strip().rstrip("?")
    m = re.search(
        r"^(?:Is|Are) (.*?) (?:a solid, a liquid, or a gas|a solid, liquid, or gas)$",
        q,
        flags=re.I,
    )
    return _norm_text(m.group(1) if m else q)


def _state_from_choice(choice):
    text = _norm_text(choice)
    if "solid" in text:
        return "solid"
    if "liquid" in text:
        return "liquid"
    if "gas" in text:
        return "gas"
    return None


def _obvious_matter_state(obj):
    # Order matters: "juice pop" is solid even though it contains "juice".
    if any(x in obj for x in [
        "juice pop", "watch", "quartz", "pair of jeans",
        "flip-flop", "rubber balloon", "coffee mug", "t-shirt", "icicle",
    ]):
        return "solid"
    if any(x in obj for x in ["air", "wind", "helium", "bubble"]):
        return "gas"
    if any(x in obj for x in ["magma", "ocean water", "egg white", "diesel", "gasoline"]):
        return "liquid"
    return None


def rule_matter_state(row):
    if row.get("skill") != MATTER_STATE_SKILL:
        return None
    state = _obvious_matter_state(_object_phrase_from_state_question(row.get("question", "")))
    if state is None:
        return None
    hits = [i for i, choice in enumerate(_choice_list(row)) if _state_from_choice(choice) == state]
    return hits[0] if len(hits) == 1 else None


ANIMAL_CLASSES = ("mammal", "bird", "fish", "reptile", "amphibian")


def _animal_target_class(question):
    q = _norm_text(question)
    for cls in ANIMAL_CLASSES:
        if cls in q:
            return cls
    return None


def _build_animal_class_lookup(known_df):
    votes = defaultdict(Counter)
    for _, row in known_df.iterrows():
        if row.get("skill") != ANIMAL_CLASS_SKILL or "answer" not in row or pd.isna(row["answer"]):
            continue
        cls = _animal_target_class(row.get("question", ""))
        if cls is None:
            continue
        votes[_norm_text(_choice_list(row)[int(row["answer"])])][cls] += 1
    lookup = {}
    for animal, counts in votes.items():
        best, n = counts.most_common(1)[0]
        if n == sum(counts.values()):
            lookup[animal] = best
    return lookup


def rule_animal_class(row, animal_lookup):
    if row.get("skill") != ANIMAL_CLASS_SKILL:
        return None
    target = _animal_target_class(row.get("question", ""))
    if target is None:
        return None
    hits = [i for i, choice in enumerate(_choice_list(row)) if animal_lookup.get(_norm_text(choice)) == target]
    return hits[0] if len(hits) == 1 else None


def _choice_numbers(row):
    vals = []
    for choice in _choice_list(row):
        m = re.search(r"-?\d+", str(choice))
        vals.append(int(m.group(0)) if m else None)
    return vals


def rule_air_mass(row, data_dir):
    if row.get("skill") != AIR_MASS_SKILL or not _HAS_SCIPY_NDIMAGE:
        return None
    img = Image.open(_resolve_image_path(data_dir, row["image_path"])).convert("RGB")
    arr = np.array(img)
    h, w = arr.shape[:2]
    rgb = arr.astype(int)

    white = (rgb.min(axis=2) > 225) & ((rgb.max(axis=2) - rgb.min(axis=2)) < 25)
    white[int(h * 0.78):, :] = False
    labels, _ = ndimage.label(white)
    comps = []
    for j, sl in enumerate(ndimage.find_objects(labels), 1):
        if sl is None:
            continue
        ys, xs = sl
        area = int((labels[sl] == j).sum())
        if area < 100:
            continue
        x0, x1 = xs.start, xs.stop - 1
        y0, y1 = ys.start, ys.stop - 1
        bw, bh = x1 - x0 + 1, y1 - y0 + 1
        if bw >= 40 and bh >= 20:
            comps.append((area, x0, x1, y0, y1, bw, bh))
    if not comps:
        return None
    _, x0, x1, y0, y1, bw, bh = max(comps, key=lambda c: c[0])

    sat = rgb.max(axis=2) - rgb.min(axis=2)
    yy_all = np.indices((h, w))[0]
    legend_mask = (sat > 35) & (yy_all > int(h * 0.75))
    cols = np.where(legend_mask.any(axis=0))[0]
    rows = np.where(legend_mask.any(axis=1))[0]
    if len(cols) == 0 or len(rows) == 0:
        return None
    lx0, lx1, ly0, ly1 = cols.min(), cols.max(), rows.min(), rows.max()
    legend = arr[ly0:ly1 + 1, lx0:lx1 + 1]
    leg_rgb = legend.astype(int)
    leg_sat = leg_rgb.max(axis=2) - leg_rgb.min(axis=2)
    row_counts = (leg_sat > 35).sum(axis=1)
    good_rows = np.where(row_counts > 0.5 * row_counts.max())[0]
    if len(good_rows) == 0:
        return None
    palette = legend[good_rows, :, :].reshape(-1, legend.shape[1], 3).mean(axis=0)

    yy, xx = np.mgrid[y0:y1 + 1, x0:x1 + 1]
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    rx, ry = bw / 2 * 0.88, bh / 2 * 0.75
    inside = (((xx - cx) / rx) ** 2 + ((yy - cy) / ry) ** 2) <= 1
    region = arr[y0:y1 + 1, x0:x1 + 1]
    reg_rgb = region.astype(int)
    reg_sat = reg_rgb.max(axis=2) - reg_rgb.min(axis=2)
    valid = inside & (reg_sat > 15) & ~(reg_rgb.min(axis=2) > 220)
    pix = region[valid]
    if len(pix) == 0:
        return None
    if len(pix) > 5000:
        pix = pix[:: max(1, len(pix) // 5000)]

    d = ((pix[:, None, :].astype(float) - palette[None, :, :].astype(float)) ** 2).sum(axis=2)
    matched_x = d.argmin(axis=1)
    is_temp = "°" in " ".join(map(str, _choice_list(row))) or "temperature" in _norm_text(row.get("question", ""))
    lo, hi = (-25, 35) if is_temp else (0, 24)
    values = lo + (hi - lo) * matched_x / max(1, len(palette) - 1)
    estimate = float(np.percentile(values, 25))
    nums = _choice_numbers(row)
    if any(v is None for v in nums):
        return None
    return min(range(len(nums)), key=lambda i: abs(nums[i] - estimate))


def _build_exact_hash_lookup(known_df, data_dir):
    lookup = defaultdict(list)
    for _, row in known_df.iterrows():
        skill = row.get("skill")
        if skill not in EXACT_HASH_SKILLS or "answer" not in row or pd.isna(row["answer"]):
            continue
        answer_text = _choice_list(row)[int(row["answer"])]
        key = (skill, _image_md5(data_dir, row["image_path"]))
        lookup[key].append(_norm_text(answer_text))
    return lookup


def rule_exact_hash(row, data_dir, exact_lookup):
    skill = row.get("skill")
    if skill not in EXACT_HASH_SKILLS:
        return None
    answers = exact_lookup.get((skill, _image_md5(data_dir, row["image_path"])), [])
    if not answers:
        return None
    counts = Counter(answers)
    answer_text, n = counts.most_common(1)[0]
    if n != len(answers):  # only trust unanimous repeats
        return None
    return _choice_index(row, answer_text)


def _food_web_question_key(question):
    q = _norm_text(question)
    for role in [
        "primary consumer", "secondary consumer", "tertiary consumer",
        "producer", "decomposer", "omnivore", "consumer",
    ]:
        if role in q:
            return ("role", role)
    m = re.search(r"eventually moves to the (.*?)$", q)
    if m:
        return ("to", m.group(1).strip("? ."))
    m = re.search(r"once part of the (.*?)$", q)
    if m:
        return ("from", m.group(1).strip("? ."))
    return ("q", q)


def _build_food_web_vote_lookup(known_df, data_dir):
    votes = defaultdict(lambda: [0, 0])  # [negative_count, positive_count]
    for _, row in known_df.iterrows():
        if row.get("skill") not in FOOD_WEB_SKILLS or "answer" not in row or pd.isna(row["answer"]):
            continue
        key_base = (_image_md5(data_dir, row["image_path"]), _food_web_question_key(row.get("question", "")))
        answer_idx = int(row["answer"])
        for i, choice in enumerate(_choice_list(row)):
            key = key_base + (_norm_text(choice),)
            votes[key][1 if i == answer_idx else 0] += 1
    return votes


def rule_food_web_vote(row, data_dir, food_web_votes):
    if row.get("skill") not in FOOD_WEB_SKILLS:
        return None
    key_base = (_image_md5(data_dir, row["image_path"]), _food_web_question_key(row.get("question", "")))
    scores = []
    for choice in _choice_list(row):
        neg, pos = food_web_votes.get(key_base + (_norm_text(choice),), [0, 0])
        scores.append(pos - neg)
    best = max(scores) if scores else None
    if best is None:
        return None
    hits = [i for i, s in enumerate(scores) if s == best]
    # This deliberately allows best == 0: often the right answer is unseen,
    # but every wrong option has been seen as a negative for the same food web.
    if len(hits) == 1:
        return hits[0]
    return None


def collect_rule_predictions(df, data_dir, known_df):
    exact_lookup = _build_exact_hash_lookup(known_df, data_dir)
    food_web_votes = _build_food_web_vote_lookup(known_df, data_dir)
    climate_lookup = _build_climate_lookup(known_df)
    animal_lookup = _build_animal_class_lookup(known_df)
    pred_by_rule = {name: [None] * len(df) for name in RULE_NAMES}
    for i, (_, row) in enumerate(tqdm(list(df.iterrows()), desc="rule candidates")):
        pred_by_rule["punnett"][i] = rule_punnett(row, data_dir)
        pred_by_rule["solution"][i] = rule_solution_concentration(row, data_dir)
        pred_by_rule["magnet"][i] = rule_magnetic_force(row, data_dir)
        pred_by_rule["particle_motion"][i] = rule_particle_motion(row, data_dir)
        pred_by_rule["planet_table"][i] = rule_planet_table(row, data_dir)
        pred_by_rule["letter_grid"][i] = rule_letter_grid(row, data_dir)
        pred_by_rule["climate_lookup"][i] = rule_climate_lookup(row, climate_lookup)
        pred_by_rule["matter_state"][i] = rule_matter_state(row)
        pred_by_rule["animal_class"][i] = rule_animal_class(row, animal_lookup)
        pred_by_rule["air_mass"][i] = rule_air_mass(row, data_dir)
        pred_by_rule["exact_hash"][i] = rule_exact_hash(row, data_dir, exact_lookup)
        pred_by_rule["food_web"][i] = rule_food_web_vote(row, data_dir, food_web_votes)
    return pred_by_rule


def choose_rule_plan(val_df, base_preds, data_dir):
    val_rules = collect_rule_predictions(val_df, data_dir, train_df)
    y = val_df["answer"].astype(int).to_numpy()
    plan = []
    current = list(base_preds)
    print("rule validation gates:")
    for name, preds in val_rules.items():
        candidate = list(current)
        changed = 0
        for i, p in enumerate(preds):
            if p is not None and 0 <= int(p) < int(val_df.iloc[i]["num_choices"]):
                candidate[i] = int(p)
                changed += 1
        before = float((np.array(current) == y).mean())
        after = float((np.array(candidate) == y).mean())
        rule_hits = [int(p) == int(y[i]) for i, p in enumerate(preds) if p is not None]
        rule_acc = float(np.mean(rule_hits)) if rule_hits else float("nan")
        print(f"  {name:10s}: covers {changed:3d}/{len(val_df)} | rule_acc={rule_acc:.4f} | {before:.4f} -> {after:.4f}")
        if after > before:
            plan.append(name)
            current = candidate
    print(f"post-rule val accuracy: {float((np.array(current) == y).mean()):.4f}")
    return plan, current


def apply_rule_plan(df, base_preds, data_dir, known_df, plan, desc="apply rules"):
    if not plan:
        return list(base_preds), {name: 0 for name in RULE_NAMES}
    all_rules = collect_rule_predictions(df, data_dir, known_df)
    final = list(base_preds)
    counts = {name: 0 for name in RULE_NAMES}
    for name in plan:
        for i, p in enumerate(all_rules[name]):
            if p is not None and 0 <= int(p) < int(df.iloc[i]["num_choices"]):
                final[i] = int(p)
                counts[name] += 1
    print(f"{desc}: rule replacements {counts}")
    return final, counts




# Sanity check on val (should be ≥ no-TTA val acc)
val_tta_base_preds = []
for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc="val (TTA)"):
    val_tta_base_preds.append(score_with_tta(model, processor, row, DATA_DIR, CHOICE_TOKEN_IDS))
val_tta_acc = (np.array(val_tta_base_preds) == val_y).mean()
print(f"val accuracy with TTA: {val_tta_acc:.4f}")

if ENABLE_RULE_OVERRIDES:
    RULE_PLAN, val_tta_preds = choose_rule_plan(val_df, val_tta_base_preds, DATA_DIR)
else:
    RULE_PLAN, val_tta_preds = [], val_tta_base_preds
val_tta_rule_acc = (np.array(val_tta_preds) == val_y).mean()
print(f"val accuracy with TTA + selected parsers: {val_tta_rule_acc:.4f}")

# Test predictions
test_preds = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="test (TTA)"):
    test_preds.append(score_with_tta(model, processor, row, DATA_DIR, CHOICE_TOKEN_IDS))

if ENABLE_RULE_OVERRIDES:
    if USE_VAL_LABELS_FOR_TEST_RULES:
        rule_known_df = pd.concat([train_df, val_df], ignore_index=True)
    else:
        rule_known_df = train_df
    test_preds, _ = apply_rule_plan(test_df, test_preds, DATA_DIR, rule_known_df,
                                    RULE_PLAN, desc="test")

submission = pd.DataFrame({"id": test_df["id"], "answer": test_preds})

# Schema validation before writing
assert list(submission.columns) == ["id", "answer"], submission.columns.tolist()
assert submission["id"].is_unique
assert submission["answer"].between(0, 4).all()
assert len(submission) == len(test_df)
for i, row in test_df.iterrows():
    assert int(submission.loc[i, "answer"]) < int(row["num_choices"]), \
        f"row {row['id']}: answer {submission.loc[i, 'answer']} >= num_choices {row['num_choices']}"

submission.to_csv("/kaggle/working/submission.csv", index=False)
print(f"wrote submission.csv with {len(submission)} rows")
print(submission.head())
